In [ ]:
!pip install -q fastapi uvicorn python-multipart albumentations pyngrok nest-asyncio

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Copy file kiến trúc model từ Drive ra môi trường tạm của Colab để import
!cp /content/drive/MyDrive/ViSign/model_resnet.py /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


TRIỂN KHAI FASTAPI SERVER

In [ ]:
import os
import time
import json

import torch
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from fastapi import FastAPI, File, UploadFile, WebSocket, WebSocketDisconnect, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import threading
from pyngrok import ngrok
from model_resnet import DETR

# Cau hinh he thong
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = "/content/drive/MyDrive/ViSign/best_signdetr_model.pth"
NUM_CLASSES = 22
DEFAULT_CONFIDENCE_THRESHOLD = 0.5

# Thu tu nhan bat buoc phai khop 100% voi file training
CLASSES = ['A', 'B', 'C', 'D', 'E', 'G', 'H', 'I', 'K', 'L',
           'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'X', 'Y']

In [ ]:
print(f"Dang nap mo hinh len {DEVICE}...")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Khong tim thay model tai: {MODEL_PATH}")

model = DETR(num_classes=NUM_CLASSES)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)

model.to(DEVICE)
model.eval()

# Cau hinh preprocessing (giu dung pipeline nhu luc train/val)
transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print("Mo hinh da san sang.")

Đang nạp mô hình lên cuda...
Mô hình đã sẵn sàng.


In [ ]:
def _run_inference(frame_bgr: np.ndarray, threshold: float = DEFAULT_CONFIDENCE_THRESHOLD):
    if frame_bgr is None or frame_bgr.size == 0:
        raise ValueError("Frame khong hop le")

    start_time = time.time()
    height, width = frame_bgr.shape[:2]

    rgb_image = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    transformed = transforms(image=rgb_image)
    input_tensor = transformed['image'].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        result = model(input_tensor)

    probabilities = result['pred_logits'].softmax(-1)[:, :, :-1]
    max_probs, max_classes = probabilities.max(-1)
    keep_mask = max_probs[0] > threshold

    detections = []
    top_prediction = None

    for i in range(keep_mask.shape[0]):
        if not keep_mask[i]:
            continue

        cls_idx = max_classes[0, i].item()
        conf = float(max_probs[0, i].item())
        bbox = result['pred_boxes'][0, i]
        cx, cy, w_box, h_box = bbox.detach().cpu().numpy()

        x1 = int((cx - w_box / 2) * width)
        y1 = int((cy - h_box / 2) * height)
        x2 = int((cx + w_box / 2) * width)
        y2 = int((cy + h_box / 2) * height)

        x1 = max(0, min(width - 1, x1))
        y1 = max(0, min(height - 1, y1))
        x2 = max(0, min(width - 1, x2))
        y2 = max(0, min(height - 1, y2))

        if x2 <= x1 or y2 <= y1:
            continue

        det = {
            "class": CLASSES[cls_idx],
            "confidence": conf,
            "bbox": [x1, y1, x2, y2]
        }
        detections.append(det)

        if top_prediction is None or conf > top_prediction["confidence"]:
            top_prediction = {"class": CLASSES[cls_idx], "confidence": conf}

    inference_ms = (time.time() - start_time) * 1000.0
    return {
        "top_prediction": top_prediction,
        "detections": detections,
        "inference_ms": round(inference_ms, 2)
    }

In [ ]:
app = FastAPI(title="ViSign API", version="1.1.0")

# Neu Flutter web can credentials=true thi thay allow_origins bang domain cu the
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    return {"status": "ok", "device": str(DEVICE)}

@app.post("/predict-image")
async def predict_image(file: UploadFile = File(...), threshold: float = DEFAULT_CONFIDENCE_THRESHOLD):
    contents = await file.read()
    if not contents:
        raise HTTPException(status_code=400, detail="File rong")

    nparr = np.frombuffer(contents, np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if frame is None:
        raise HTTPException(status_code=400, detail="Khong decode duoc anh")

    if threshold < 0.0 or threshold > 1.0:
        raise HTTPException(status_code=400, detail="threshold phai trong [0, 1]")

    try:
        result = _run_inference(frame, threshold=threshold)
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Loi suy luan: {e}")

@app.websocket("/ws/predict")
async def ws_predict(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            frame_bytes = await websocket.receive_bytes()
            nparr = np.frombuffer(frame_bytes, np.uint8)
            frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

            if frame is None:
                await websocket.send_text(json.dumps({
                    "ok": False,
                    "error": "Khong decode duoc frame"
                }))
                continue

            try:
                result = _run_inference(frame, threshold=DEFAULT_CONFIDENCE_THRESHOLD)
                payload = {
                    "ok": True,
                    "timestamp": int(time.time() * 1000),
                    **result
                }
                await websocket.send_text(json.dumps(payload))
            except Exception as e:
                await websocket.send_text(json.dumps({
                    "ok": False,
                    "error": f"Loi suy luan: {e}"
                }))
    except WebSocketDisconnect:
        print("WebSocket disconnected")

EXPOSE PUBLIC URL BẰNG NGROK

In [ ]:
# Dat token vao Colab secret/environment: NGROK_AUTH_TOKEN
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN")
if not NGROK_AUTH_TOKEN:
    raise ValueError("Ban chua set NGROK_AUTH_TOKEN trong environment")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

# Bo domain de pyngrok cap domain moi moi lan, on dinh hon voi free plan
ngrok_tunnel = ngrok.connect(8000)
print("Public URL:", ngrok_tunnel.public_url)
print("Health:", f"{ngrok_tunnel.public_url}/health")

def run_server():
    import nest_asyncio
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

# Chay server trong luong rieng de khong treo Colab
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

Public URL: https://apiaceous-penelope-halterlike.ngrok-free.dev
